In [2]:
from dataclasses import dataclass

@dataclass
class Ride:
    PULocationID: int
    DOLocationID: int
    trip_distance: float
    total_amount: float
    lpep_pickup_datetime: int

In [3]:
import json

def ride_deserializer(data):
    json_str = data.decode('utf-8')
    ride_dict = json.loads(json_str)
    return Ride(**ride_dict)

In [10]:
test_bytes = json.dumps({
    'PULocationID': 186,
    'DOLocationID': 79,
    'trip_distance': 1.72,
    'total_amount': 17.31,
    'lpep_pickup_datetime': 1730429702000
}).encode('utf-8')

ride_deserializer(test_bytes)
# Ride(PULocationID=186, DOLocationID=79, trip_distance=1.72,
#      total_amount=17.31, tpep_pickup_datetime=1730429702000)

Ride(PULocationID=186, DOLocationID=79, trip_distance=1.72, total_amount=17.31, lpep_pickup_datetime=1730429702000)

In [4]:
from kafka import KafkaConsumer

server = 'localhost:9092'
topic_name = 'rides'

consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-console',
    value_deserializer=ride_deserializer
)

In [5]:
from datetime import datetime

print(f"Listening to {topic_name}...")

count = 0
for message in consumer:
    ride = message.value
    pickup_dt = datetime.fromtimestamp(ride.lpep_pickup_datetime / 1000)
    print(f"Received: PU={ride.PULocationID}, DO={ride.DOLocationID}, "
          f"distance={ride.trip_distance}, amount=${ride.total_amount:.2f}, "
          f"pickup={pickup_dt}")
    count += 1
    if count >= 10:
        print(f"\n... received {count} messages so far (stopping after 10 for demo)")
        break

consumer.close()

Listening to rides...
Received: PU=247, DO=69, distance=0.7, amount=$10.00, pickup=2025-10-01 08:21:47
Received: PU=247, DO=69, distance=0.7, amount=$10.00, pickup=2025-10-01 08:21:47
Received: PU=247, DO=69, distance=0.7, amount=$10.00, pickup=2025-10-01 08:21:47
Received: PU=66, DO=25, distance=1.61, amount=$16.68, pickup=2025-10-01 08:14:03
Received: PU=244, DO=244, distance=0.0, amount=$13.20, pickup=2025-10-01 08:16:44
Received: PU=95, DO=170, distance=10.37, amount=$67.85, pickup=2025-10-01 08:07:36
Received: PU=82, DO=138, distance=4.07, amount=$34.12, pickup=2025-10-01 05:10:29
Received: PU=129, DO=37, distance=7.13, amount=$47.12, pickup=2025-10-01 05:49:46
Received: PU=95, DO=134, distance=1.13, amount=$13.88, pickup=2025-10-01 08:11:24
Received: PU=95, DO=70, distance=6.01, amount=$34.32, pickup=2025-10-01 08:07:08

... received 10 messages so far (stopping after 10 for demo)
